# Causal Workbench - Phase 1

Team T094, Section E. Classical machine learning, reviewed text extraction and a separate synthetic Bayesian system.

Use the project’s Python 3.12 environment. Raw data is included. No neural model or API key is required.

In [ ]:
from pathlib import Path
import sys,os,json
import pandas as pd
from IPython.display import display,Image
ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
from src.experiment import load_splits,run
from src.causal import extract,build_graph,fitted_network,query,oracle
print("Project:",ROOT.name)

## Source and split audit

We use the first 12,000 official training rows and all development/test rows at a pinned revision. Exact premise overlap is removed before features are fitted. This prefix limits representativeness.

In [ ]:
display(json.loads((ROOT/"data/raw/manifest.json").read_text()))
splits,audit=load_splits()
display(pd.DataFrame(audit).T)
assert not set(splits["train"].group)&set(splits["test"].group)
display(pd.read_csv(ROOT/"artifacts/eda_summary.csv"))
display(Image(filename=str(ROOT/"artifacts/eda.png")))

## Train classical models

Word unigrams and bigrams retain single-letter variable names. TF-IDF fits only on train. We choose among the main models using development macro F1 and report the unigram ablation separately.

In [ ]:
metrics=run()
display(pd.DataFrame({k:v["test"] for k,v in metrics["results"].items()}).T)
print("Selected:",metrics["selected_model"])
print("Group-bootstrap interval:",metrics["test_macro_f1_group_bootstrap_95ci"])

## Inspect errors and feature associations

Weights describe label associations, not identified causal mechanisms. Macro F1 complements positive precision and recall under class imbalance.

In [ ]:
from scripts.analyze import main as analyze
analyze()
display(pd.read_csv(ROOT/"artifacts/template_metrics.csv"))
display(pd.read_csv(ROOT/"artifacts/feature_weights.csv").head(10))
display(pd.read_csv(ROOT/"artifacts/error_examples.csv")[["input","label","prediction","probability"]].head(3))

## Review text relations and construct a DAG

This is an explicit-pattern demonstration, separate from Corr2Cause reasoning. In the app, all candidates start unapproved. Here we explicitly approve the fixed demonstration candidates, then inspect the cycle rejection.

In [ ]:
text="Rain causes wet grass. Sprinkler causes wet grass. Rain causes sprinkler. Wet grass causes rain."
candidates=extract(text,"notebook-demo")
display(candidates)
graph,accepted,rejected=build_graph(candidates.to_dict("records"))
display(pd.DataFrame(accepted));display(pd.DataFrame(rejected))
import networkx as nx
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(7,4))
nx.draw_networkx(graph,pos=nx.spring_layout(graph,seed=42),ax=ax,node_size=2200,node_color="#b7c5bb",arrowsize=20)
ax.axis("off");plt.show()

## Component validation

The 16 authored extraction documents are regression checks, not an independent annotated scientific/news corpus.

In [ ]:
from scripts.validate_components import main as validate
validate()
checks=json.loads((ROOT/"artifacts/component_validation.json").read_text())
display(checks["extraction"]);display(checks["graph"])

## Bayesian parameters and interventions

The declared synthetic DAG contains Rain to Sprinkler, Rain to Wet grass, and Sprinkler to Wet grass. Six thousand seeded observations supply CPDs under BDeu smoothing. Intervening cuts incoming edges. The fixed probability model is separate from the reviewed text graph. We do not claim individual counterfactual inference.

In [ ]:
model=fitted_network()
assert model.check_model()
for cpd in model.get_cpds():print(cpd)
rows=[]
for s in [0,1]:
    observed=query(value=s)
    intervention=query(value=s,intervention=True)
    truth=oracle(s)
    assert abs(intervention-truth)<.035
    rows.append({"sprinkler":s,"observed":observed,"do_estimate":intervention,"oracle":truth,"absolute_error":abs(intervention-truth)})
display(pd.DataFrame(rows))

## Conclusions

The selected classical baseline improves macro F1 over majority prediction. Positive precision remains limited. A bounded training prefix and only 18 exact test premise groups constrain conclusions. The lexical parser misses richer constructions, and the causal demonstration assumes a known synthetic mechanism. See the report for assumptions and sources.